# Product Reorder Prediction 


## 1. Import libraries

In [ ]:
import pathlib

import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
# Resolve paths relative to this notebook's location.
# Jupyter (and VS Code notebooks) set the CWD to the notebook's folder.
_NB_DIR = pathlib.Path.cwd()
_DATASET_DIR = _NB_DIR.parent / "Modelling" / "ProdPred" / "Dataset"

## 2. Load and merge the data

Here the four CSV files are loaded and combined into one working dataframe for modelling.

The merged `data` table links:
- each order
- the products in that order
- the user who placed it
- the product category

In [ ]:
# Load data
users = pd.read_csv(_DATASET_DIR / "users.csv")
products = pd.read_csv(_DATASET_DIR / "products.csv")
orders = pd.read_csv(_DATASET_DIR / "orders.csv")
order_products = pd.read_csv(_DATASET_DIR / "order_products.csv")

# Merge everything
data = order_products.merge(orders, on="order_id") \
                     .merge(products, on="product_id")

In [ ]:
print("Merged dataset shape:", data.shape)
display(data.head())

## 3. Create model features

This section builds the features used by the recommendation model:

- `user_total_orders`: how many orders a user has placed
- `product_total_purchases`: how often a product has been bought overall
- `product_reorder_rate`: how often a product is reordered
- `user_product_purchase_count`: how many times a specific user has bought a specific product

In [ ]:
# User-level feature
user_order_counts = orders.groupby("user_id")["order_id"].count()
data["user_total_orders"] = data["user_id"].map(user_order_counts)

# Product-level features
product_purchase_counts = data.groupby("product_id").size()
product_reorder_rates = data.groupby("product_id")["reordered"].mean()

data["product_total_purchases"] = data["product_id"].map(product_purchase_counts)
data["product_reorder_rate"] = data["product_id"].map(product_reorder_rates)

# User–product interaction feature
user_product_counts = (
    data.groupby(["user_id", "product_id"])
        .size()
        .rename("user_product_purchase_count")
        .reset_index()
)

data = data.merge(
    user_product_counts,
    on=["user_id", "product_id"],
    how="left"
)


In [ ]:
display(
    data[
        [
            "user_id",
            "product_id",
            "category",
            "reordered",
            "user_total_orders",
            "product_total_purchases",
            "product_reorder_rate",
            "user_product_purchase_count",
        ]
    ].head()
)

## 3b. Engineer three additional features (v2)

Two additional features are engineered to improve model expressiveness:

**Feature 1 — `days_since_last_purchase`**  
Approximated from `order_number` within each user's history. The most recent order (highest `order_number` for that user) has 0 days; older orders are proportionally further back (scaled to a 0–365 window).

**Feature 2 — `organic_preference`**  
25% of products are flagged organic. Per-user organic preference is the fraction of that user's purchases that were organic-certified.

> Seasonal availability is handled as a backend availability filter — out-of-season products are excluded before the model scores them.

In [ ]:
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# Normalise order_number to [0, 1] within each user (0 = oldest, 1 = newest)
max_order_per_user = orders.groupby("user_id")["order_number"].max()
data["max_order_number"] = data["user_id"].map(max_order_per_user)
data["order_recency_frac"] = (
    data["order_number"] / data["max_order_number"].clip(lower=1)
)
# Days since: newest purchase → 0 days, oldest → up to 365 days
data["days_since_last_purchase"] = (
    (1.0 - data["order_recency_frac"]) * 365
).astype(int)

unique_pids = products["product_id"].unique()
organic_pids = set(
    rng.choice(unique_pids, size=int(len(unique_pids) * 0.25), replace=False)
)
products["is_organic"] = products["product_id"].isin(organic_pids).astype(int)
data = data.merge(
    products[["product_id", "is_organic"]],
    on="product_id",
    how="left",
    suffixes=("", "_dup"),
)

user_organic_pref = (
    data.groupby("user_id")["is_organic"].mean().rename("organic_preference")
)
data["organic_preference"] = data["user_id"].map(user_organic_pref)

print("New features engineered:")
print("  days_since_last_purchase:", data["days_since_last_purchase"].describe().to_dict())
print("  organic_preference (mean):", data["organic_preference"].mean())

## 4. Modelling data

The feature matrix `X` contains the predictor variables, and `y` is the target variable that indicates whether a product was reordered.

In [ ]:
features = [
    "user_total_orders",
    "product_total_purchases",
    "product_reorder_rate",
    "user_product_purchase_count",
    "days_since_last_purchase",
    "organic_preference",
    "category",
]

X = data[features]
y = data["reordered"]

In [ ]:
print("Feature columns:")
print(features)
print("\nTarget distribution:")
display(y.value_counts(normalize=True).rename("proportion"))

## 5. Build the preprocessing and modelling pipeline

The pipeline:
- passes numerical features through unchanged
- one-hot encodes the product category
- trains a logistic regression classifier

This keeps preprocessing and model training together in one workflow.

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_features = [
    "user_total_orders",
    "product_total_purchases",
    "product_reorder_rate",
    "user_product_purchase_count",
    "days_since_last_purchase",
    "organic_preference",
]
categorical_features = ["category"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

## 6. Data Splitting and Model Training

The data is split into training and test sets, then the logistic regression pipeline is fitted.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)

print("Model trained successfully.")


## 7. Evaluate model performance

These cells assess how well the model performs on the test set using:
- accuracy
- precision
- recall
- F1-score
- ROC-AUC
- confusion matrix
- ROC curve

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report
)

# Predictions on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Evaluation Metrics")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\nClassification Report")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Confusion Matrix")
plt.show()

# ROC curve
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve")
plt.show()

### Probability distribution plot

This plot helps show how confidently the model separates reordered and non-reordered examples.

In [ ]:
# Predicted probability distributions
results_df = pd.DataFrame({
    "actual_reordered": y_test.values,
    "predicted_probability": y_prob
})

plt.figure(figsize=(8, 5))

plt.hist(
    results_df[results_df["actual_reordered"] == 0]["predicted_probability"],
    bins=30,
    alpha=0.6,
    label="Not Reordered"
)

plt.hist(
    results_df[results_df["actual_reordered"] == 1]["predicted_probability"],
    bins=30,
    alpha=0.6,
    label="Reordered"
)

plt.xlabel("Predicted Reorder Probability")
plt.ylabel("Frequency")
plt.title("Distribution of Predicted Probabilities")
plt.legend()
plt.show()

### Coefficient inspection

Because the model is logistic regression, the fitted coefficients can be inspected to understand which features push predictions up or down.

In [ ]:
# Get feature names after preprocessing
feature_names = model.named_steps["preprocessor"].get_feature_names_out()

# Get logistic regression coefficients
coefficients = model.named_steps["classifier"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

coef_df = coef_df.sort_values("coefficient", ascending=False)

print("\nTop Positive Features")
print(coef_df.head(10))

print("\nTop Negative Features")
print(coef_df.tail(10))

# Plot top coefficients
top_features = pd.concat([coef_df.head(10), coef_df.tail(10)])

plt.figure(figsize=(10, 6))
plt.barh(top_features["feature"], top_features["coefficient"])
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.title("Top Logistic Regression Coefficients")
plt.tight_layout()
plt.show()

### Ranked predicted probabilities

This gives a quick view of how the model's predicted reorder probabilities are distributed across the test samples.

In [ ]:
# Sort by predicted probability
sorted_probs = np.sort(y_prob)[::-1]

plt.figure(figsize=(8, 5))
plt.plot(sorted_probs)
plt.xlabel("Test Sample Rank")
plt.ylabel("Predicted Reorder Probability")
plt.title("Ranked Predicted Reorder Probabilities")
plt.show()

## 8. Prototype recommendation function

This function scores every product for a given user and returns the products with the highest predicted reorder probabilities.

In [ ]:
def recommend_products_for_user(user_id, top_n=5):
    user_data = products.copy()

    user_data["user_id"] = user_id

    user_total_orders = orders[orders["user_id"] == user_id].shape[0]
    user_data["user_total_orders"] = user_total_orders

    user_product_counts = (
        data[data["user_id"] == user_id]
        .groupby("product_id")
        .size()
        .to_dict()
    )

    user_data["user_product_purchase_count"] = user_data["product_id"].map(
        user_product_counts
    ).fillna(0)

    user_data["product_total_purchases"] = user_data["product_id"].map(
        product_purchase_counts
    )

    user_data["product_reorder_rate"] = user_data["product_id"].map(
        product_reorder_rates
    )

    # days_since_last_purchase: use the user's most recent order recency
    user_rows = data[data["user_id"] == user_id]
    if not user_rows.empty:
        user_data["days_since_last_purchase"] = int(user_rows["days_since_last_purchase"].min())
    else:
        user_data["days_since_last_purchase"] = 365

    # organic_preference: look up from the computed per-user value
    user_organic = data[data["user_id"] == user_id]["organic_preference"]
    user_data["organic_preference"] = float(user_organic.iloc[0]) if not user_organic.empty else 0.0

    X_user = user_data[features]

    user_data["reorder_probability"] = model.predict_proba(X_user)[:, 1]

    recommendations = user_data.sort_values(
        "reorder_probability", ascending=False
    ).head(top_n)

    return recommendations[
        ["product_id", "category", "reorder_probability"]
    ]

### Example recommendation

An example is shown below for `user_id=60`.

In [ ]:
print(recommend_products_for_user(user_id=60, top_n=5))


## 9. Export model 

These cells save the trained model and the supporting data needed later for inference in a separate script or application.

In [ ]:
# Save locally in this notebook's directory (LogModel/)
joblib.dump(model)
print(f"Model saved to directory.")
print(f"To deploy: copy files to recommender/artifacts/")

joblib.dump(product_purchase_counts, _NB_DIR / "product_purchase_counts.joblib")
joblib.dump(product_reorder_rates, _NB_DIR / "product_reorder_rates.joblib")

print("Export complete.")

In [ ]:
user_product_counts = (
    data.groupby(["user_id", "product_id"])
        .size()
        .rename("user_product_purchase_count")
        .reset_index()
)

user_product_counts.to_csv(
    _DATASET_DIR / "user_product_purchase_counts.csv",
    index=False
)